# Teacher setup — Particle Physics Lab

End-to-end demo of the Cadence teaching workflow. **Three notebooks are involved** — read this first so the file flow is clear:

```
(1) teacher.ipynb              ← what you write: your worked solutions
        │
        │  %cadence_autoregister   (you run this once)
        ▼
(2) teacher_registered.ipynb   ← auto-generated: same notebook + registration magic
        │
        │  %cadence_scaffold       (already wired in at the bottom; runs as part of "Run All")
        ▼
(3) teacher_registered_student.ipynb   ← what you share with students
```

You only ever **author** the first file. The other two are derived.

## The rules in 60 seconds

**You do not have to mark anything.** With zero markers, `%cadence_autoregister` runs in *auto mode*: every markdown-heading cell paired with the code cell that follows it becomes an exercise. Pure-import cells, setup cells (e.g. `rng = np.random.default_rng(7)`), and `%magic`-only cells are skipped and copied through verbatim. So if your notebook is just `## Heading → code → ## Heading → code …`, nothing else needed — run autoregister and you're done.

Markers exist for **per-cell control** when auto mode isn't enough. All markers are comments (inert at runtime); only `%magic` lines do anything when you run them. Add markers selectively:

| Want to… | Use |
|---|---|
| Set a custom checkpoint id, override comparator, or make it free-text | `# cadence:checkpoint <id> [<comparator>]` on the exercise cell |
| Add a hint | `# cadence:hint: <text>` on a line inside the exercise cell |
| Give students a scaffolded starting point instead of a blank cell | Wrap a region in `# cadence:starter` / `# cadence:end` |
| Hide a teacher-only authoring note from both registered and student notebooks | `# cadence:hide` / `# cadence:end` (or the `<!-- ... -->` form in markdown) |
| Share a code cell with students verbatim (worked solution, helper, etc.) | `# cadence:solution` on the cell |

Mix freely: you can have some cells auto-detected and others tagged with explicit markers in the same notebook.

The demo below shows all of them. Each exercise has a comment naming which marker(s) it's using and why.

## Setup

In [ ]:
# No marker needed — this is a pure setup cell. Autoregister sees the
# heading above but the value (a numpy Generator) isn't a primitive
# answer type, so it's treated as setup and copied through verbatim.
import numpy as np
rng = np.random.default_rng(1)

## Part 1 — NumPy warmup

### Exercise 1 — Mean of `np.arange(100)`
Compute the mean and bind it to `mean_value`.

In [ ]:
# Pure auto mode — no marker on this cell. Heading above + numeric
# answer below = autoregister picks it up. The id will be slugged from
# the heading: `exercise-1-mean-of-nparange100`. Fine for a demo;
# add an explicit marker if you want a tidier id.
mean_value = np.arange(100).mean()

### Exercise 2 — Row sums
Sum each row of `np.arange(12).reshape(3, 4)` and return as a list.

In [ ]:
# Explicit marker — gives this checkpoint a stable, hand-picked id
# instead of the slugged heading. Also adds an inline hint with
# markdown formatting.
# cadence:checkpoint setup.row-sums
# cadence:hint: `M.sum(axis=1).tolist()`
row_sums = np.arange(12).reshape(3, 4).sum(axis=1).tolist()

## Part 2 — Particle four-vectors

### Exercise 3 — Find the Higgs peak
Given simulated diphoton invariant masses below, find the integer-GeV bin centre with the most events.

<!-- cadence:hide -->
*Teacher note: this is where most students get stuck. The starter block in the code cell below scaffolds the histogram → argmax structure so they don't stare at a blank cell. The student won't see this aside (it's wrapped in `cadence:hide`).*
<!-- cadence:end -->

In [ ]:
# Explicit marker with comparator override — `exact` forces an integer
# match. cadence:starter / cadence:end scaffolds the student cell with
# the Step 1 / Step 2 structure (otherwise they'd get `# Your code here`).
# cadence:checkpoint discovery.higgs-peak exact
# cadence:hint: `np.histogram(..., bins=np.arange(100, 151))` then `argmax`
background = rng.uniform(100.0, 150.0, size=450)
signal     = rng.normal(loc=125.0, scale=1.5, size=80)
m_gg = np.concatenate([background, signal])

# cadence:starter
# Step 1 — histogram with 1-GeV bins from 100 to 150
bin_edges = np.arange(100, 151)
counts = ...

# Step 2 — peak bin
peak_idx = ...
answer = int(bin_edges[peak_idx])
# cadence:end

# Teacher reference (kept here so the cell self-tests; stripped from
# the student notebook because it's outside the cadence:starter block):
counts, _ = np.histogram(m_gg, bins=bin_edges)
answer = int(bin_edges[int(np.argmax(counts))])

### Exercise 4 — Reflection
What does the peak shape tell you about the signal-to-background ratio? Write 2–3 sentences.

In [ ]:
# `manual` comparator = free-text answer. No value extraction; the
# student writes prose in the cell and calls `mark_done("...")` to
# attest. Good for reflections, plot reviews, anything qualitative.
# cadence:checkpoint discovery.reflect manual
# cadence:hint: think about width, height vs background level

## Now wire it up

1. **Run every cell above end-to-end** (Cell → Run All Above, or just Run All from the top). This puts `mean_value`, `row_sums`, `answer` in the live kernel — autoregister reads the values from there.
2. **Run the cell below.** It prompts you through four questions:
   - Reveal solutions after N attempts?
   - Sign in to track under your account?
   - Add to a course? (only asked if signed in)
   - How many days to retain student data? (empty = 7 standalone / 90 in a course)
3. It writes `<this-notebook>_registered.ipynb`. **Open that file** in Jupyter, hit **Run All**, and the final `%cadence_scaffold` cell (added for you) writes `<this-notebook>_registered_student.ipynb`. That's the file you share with students.

You can re-run autoregister anytime — it's idempotent. Editing this notebook then re-running regenerates both downstream files cleanly.

In [ ]:
%load_ext cadence
%cadence_autoregister

## Alternative registration paths

If `%cadence_autoregister` doesn't fit your workflow, the original entry points still work — see the [README's *Alternative registration paths* section](https://github.com/livaage/cadence/tree/main/jupyter-integration#alternative-registration-paths) for the full reference. Briefly:

- **`%%cadence_register_yaml`** in a cell — register many checkpoints from one explicit YAML block. Good when you want to type expected values rather than have them inferred.
- **`%cadence_register_yaml_file checkpoints.yaml`** — same, but the YAML lives in version control alongside the notebook.
- **`%cadence_register <id> --comparator ... --expected '...'`** — one inline magic per checkpoint. Good for surgical edits or one-off demos.
- **`cadence.api.CadenceAPI().register_checkpoint(...)`** — Python API, for generating checkpoints programmatically.